# Gold annotate v2 — Bloque B

Todo el código va **en este notebook** (no hace falta subir `annotate_gold_chunks.py`).

### Colab — solo sube 1 archivo a Drive

`docs/rag_eval_queries.json` (desde tu PC) →  
`MyDrive/RAG_UPC_Final_project/docs_queries/rag_eval_queries.json`

Chroma: `.../chroma_db/`

1. `WRITE_GOLD = False` → ejecutar → revisar tabla
2. `WRITE_GOLD = True` → re-ejecutar celda de anotación
3. Descargar gold actualizado al repo local
4. Bloque A → `graph_rag_eval_v2.ipynb`

In [6]:
%pip install -q chromadb pandas

import json
import re
import sys
import unicodedata
from pathlib import Path

import chromadb
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
WRITE_GOLD = False
MIN_SCORE = 0.35
COLLECTION_NAME = "cosora_actas_e5"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/RAG_UPC_Final_project")
else:
    DRIVE_ROOT = Path("../../data").resolve()

DRIVE_QUERIES_DIR = DRIVE_ROOT / "docs_queries"
CHROMA_PATH = DRIVE_ROOT / "chroma_db"
GOLD_PATH = DRIVE_QUERIES_DIR / "rag_eval_queries.json"
REPORT_PATH = DRIVE_QUERIES_DIR / "gold_chunk_annotation_report.json"
DRIVE_QUERIES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Colab={IN_COLAB}  WRITE_GOLD={WRITE_GOLD}  MIN_SCORE={MIN_SCORE}")
print(f"Chroma: {CHROMA_PATH}")
print(f"Gold:   {GOLD_PATH}")
if not GOLD_PATH.exists():
    raise FileNotFoundError(
        f"Sube rag_eval_queries.json v2 a:\n  {GOLD_PATH.parent}/"
    )
if not CHROMA_PATH.exists():
    raise FileNotFoundError(f"No existe Chroma: {CHROMA_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab=True  WRITE_GOLD=False  MIN_SCORE=0.35
Chroma: /content/drive/MyDrive/RAG_UPC_Final_project/chroma_db
Gold:   /content/drive/MyDrive/RAG_UPC_Final_project/docs_queries/rag_eval_queries.json


In [7]:
def norm_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", text.lower().strip())


def norm_doc_stem(name: str | None) -> str:
    return Path(name).stem.lower() if name else ""


def extract_terms(*texts: str, min_len: int = 4, max_terms: int = 14) -> list[str]:
    stop = {
        "para", "como", "sobre", "desde", "hasta", "donde", "durante", "toda",
        "todo", "todos", "cada", "esta", "este", "estos", "estas", "con", "sin",
        "por", "que", "del", "las", "los", "una", "uno", "son", "hay", "fue",
        "ser", "sus", "ese", "esa", "conforme",
    }
    terms: list[str] = []
    for text in texts:
        for tok in re.findall(r"\w+", norm_text(text or "")):
            if len(tok) < min_len or tok in stop:
                continue
            if tok not in terms:
                terms.append(tok)
            if len(terms) >= max_terms:
                return terms
    return terms


def score_chunk(text: str, terms: list[str]) -> float:
    body = norm_text(text.removeprefix("passage: "))
    return sum(1 for t in terms if t in body) / len(terms) if terms else 0.0


def chunk_preview(text: str, limit: int = 120) -> str:
    body = text.removeprefix("passage: ").replace("\n", " ")
    return body[:limit] + ("…" if len(body) > limit else "")


def find_best_chunk(docs, metas, *, doc_stem: str, terms: list[str]):
    best_score, best_id, best_preview = 0.0, None, ""
    for doc, meta in zip(docs, metas):
        if doc_stem and norm_doc_stem(meta.get("doc_id", "")) != doc_stem:
            continue
        sc = score_chunk(doc, terms)
        if sc > best_score:
            best_score = sc
            best_id = meta.get("chunk_id")
            best_preview = chunk_preview(doc)
    return best_id, best_score, best_preview


def annotate_single_queries(queries, docs, metas, *, min_score: float):
    rows = []
    for q in queries:
        if q.get("eval_mode", "single_doc") not in ("single_doc", "single_chunk"):
            continue
        if q.get("source_chunk_id"):
            rows.append({"id": q["id"], "status": "skip", "source_chunk_id": q["source_chunk_id"]})
            continue
        terms = extract_terms(q.get("expected_answer", ""), q.get("query", ""))
        best_id, best_score, preview = find_best_chunk(
            docs, metas, doc_stem=norm_doc_stem(q.get("source_doc")), terms=terms
        )
        accepted = bool(best_id and best_score >= min_score)
        rows.append({
            "id": q["id"],
            "source_doc": q.get("source_doc"),
            "terms": terms[:8],
            "proposed_chunk_id": best_id,
            "score": round(best_score, 3),
            "accepted": accepted,
            "preview": preview,
        })
        print(f"{'✓' if accepted else '?'} {q['id']}: {best_id or '—'}  score={best_score:.2f}")
    return rows


def validate_any_doc(row, docs, metas):
    terms = extract_terms(row.get("expected_answer", ""), row.get("query", ""))
    per_doc = []
    for doc in row.get("source_docs") or []:
        cid, score, preview = find_best_chunk(docs, metas, doc_stem=norm_doc_stem(doc), terms=terms)
        per_doc.append({"source_doc": doc, "best_chunk_id": cid, "score": round(score, 3), "preview": preview})
    ok = sum(1 for d in per_doc if d["score"] >= 0.25)
    return {
        "id": row["id"],
        "eval_mode": row.get("eval_mode"),
        "n_source_docs": len(per_doc),
        "n_docs_with_signal": ok,
        "per_doc": per_doc,
        "ok": ok >= 1,
    }


def apply_annotations(queries, rows, *, min_score: float) -> int:
    by_id = {r["id"]: r for r in rows if r.get("proposed_chunk_id")}
    n = 0
    for q in queries:
        row = by_id.get(q["id"])
        if not row or not row.get("accepted") or row["score"] < min_score:
            continue
        q["source_chunk_id"] = row["proposed_chunk_id"]
        q["eval_mode"] = "single_chunk"
        q["notes"] = f"Anotado auto score={row['score']} — revisar manualmente"
        n += 1
    return n


print("✅ Funciones de anotación listas (inline)")

✅ Funciones de anotación listas (inline)


In [8]:
wrapper = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
assert wrapper.get("schema_version", 0) >= 2, "Gold debe ser schema v2"
queries = wrapper["queries"]

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
col = client.get_collection(COLLECTION_NAME)
_res = col.get(include=["documents", "metadatas"])
ALL_DOCS, ALL_METAS = _res["documents"], _res["metadatas"]
print(f"✅ Chroma {len(ALL_DOCS)} chunks  |  Gold {len(queries)} queries")

print("\n=== Q1–Q17: proponer source_chunk_id ===")
single_rows = annotate_single_queries(queries, ALL_DOCS, ALL_METAS, min_score=MIN_SCORE)

print("\n=== Q18–Q21: validar source_docs ===")
multi_rows = []
for q in queries:
    if q.get("eval_mode") not in ("any_doc", "cross_doc_aggregate"):
        continue
    vr = validate_any_doc(q, ALL_DOCS, ALL_METAS)
    multi_rows.append(vr)
    print(
        f"{'✓' if vr['ok'] else '✗'} {vr['id']}: "
        f"{vr['n_docs_with_signal']}/{vr['n_source_docs']} actas con señal"
    )

report = {
    "gold_path": str(GOLD_PATH),
    "n_chunks": len(ALL_DOCS),
    "min_score": MIN_SCORE,
    "single_doc_annotations": single_rows,
    "multi_doc_validation": multi_rows,
    "write_gold": WRITE_GOLD,
}
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"\n📄 Reporte → {REPORT_PATH}")

if WRITE_GOLD:
    n = apply_annotations(queries, single_rows, min_score=MIN_SCORE)
    wrapper["queries"] = queries
    GOLD_PATH.write_text(json.dumps(wrapper, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"✅ Gold actualizado ({n} → single_chunk) → {GOLD_PATH}")
else:
    print("\n(dry-run — pon WRITE_GOLD=True en celda 1 y re-ejecuta esta celda)")

✅ Chroma 1544 chunks  |  Gold 23 queries

=== Q1–Q17: proponer source_chunk_id ===
✓ Q1: 254275-DO-AVO-16-V07__c0005  score=0.86
✓ Q2: 254275-DO-AVO-15-V07__c0002  score=0.62
✓ Q3: 254275-DO-AVO-17-V07__c0001  score=0.93
? Q4: —  score=0.00
✓ Q5: 254275-DO-AVO-22-V01__c0008  score=0.79
✓ Q6: 254275-DO-AVO-22-V01__c0010  score=0.93
✓ Q7: 254275-DO-AVO-22-V01__c0004  score=0.57
✓ Q8: 254275-DO-AVO-22-V01__c0003  score=0.79
✓ Q9: 254275-DO-AVO-16-V07__c0005  score=0.64
✓ Q10: 254275-DO-AVO-22-V01__c0007  score=0.86
✓ Q11: 254275-DO-AVO-17-V07__c0002  score=0.83
✓ Q12: 254275-DO-AVO-17-V07__c0002  score=0.86
? Q13: —  score=0.00
✓ Q14: 254275-DO-AVO-16-V07__c0001  score=0.64
? Q15: 254275-DO-AVO-22-V01__c0004  score=0.33
✓ Q16: 254275-DO-AVO-22-V01__c0005  score=0.93
✓ Q17: 254275-DO-AVO-22-V01__c0006  score=0.86

=== Q18–Q21: validar source_docs ===
✓ Q18: 3/3 actas con señal
✓ Q19: 2/3 actas con señal
✗ Q20: 0/3 actas con señal
✓ Q21: 2/2 actas con señal

📄 Reporte → /content/drive/MyDri

In [9]:
df = pd.DataFrame(report["single_doc_annotations"])
cols = [c for c in ["id", "score", "accepted", "proposed_chunk_id", "preview"] if c in df.columns]
print("=== Q1–Q17: revisar antes de WRITE_GOLD=True ===")
display(df[cols])

mdf = pd.DataFrame([
    {"id": r["id"], "ok": r["ok"], "docs_con_señal": f"{r['n_docs_with_signal']}/{r['n_source_docs']}"}
    for r in report["multi_doc_validation"]
])
print("\n=== Q18–Q21 ===")
display(mdf)

=== Q1–Q17: revisar antes de WRITE_GOLD=True ===


,id,score,accepted,proposed_chunk_id,preview
0,Q1,0.857,True,254275-DO-AVO-16-V07__c0005,. Último tramo desde la zona central del andén...
1,Q2,0.615,True,254275-DO-AVO-15-V07__c0002,. Inicio de los trabajos de la rampa de final ...
2,Q3,0.929,True,254275-DO-AVO-17-V07__c0001,. || AVANCE DE LA OBRA: 1.2. Breve Reportaje f...
3,Q4,0.000,False,None,
4,Q5,0.786,True,254275-DO-AVO-22-V01__c0008,. Desalineación de pletina en pilar P07 Durant...
5,Q6,0.929,True,254275-DO-AVO-22-V01__c0010,. En función del resultado de este análisis: P...
6,Q7,0.571,True,254275-DO-AVO-22-V01__c0004,". Sanear zona de andén, (donde producido el de..."
7,Q8,0.786,True,254275-DO-AVO-22-V01__c0003,. Ejecución de arquetas. 2.2. Planificación se...
8,Q9,0.636,True,254275-DO-AVO-16-V07__c0005,. Último tramo desde la zona central del andén...
9,Q10,0.857,True,254275-DO-AVO-22-V01__c0007,. 3. || TEMAS DE OBRA: 3.1. Montaje pilares es...



=== Q18–Q21 ===


,id,ok,docs_con_señal
0,Q18,True,3/3
1,Q19,True,2/3
2,Q20,False,0/3
3,Q21,True,2/2


## Después

- Descarga `docs_queries/rag_eval_queries.json` de Drive → repo local `docs/rag_eval_queries.json`
- Bloque A: `graph_rag_eval_v2.ipynb` (gold ya en Drive)

## 5. Validar gold vs Chroma (opcional)

Ejecutar **después** de la celda 3 (`col`, `wrapper` en memoria). Comprueba actas y chunks del gold.

In [11]:
def chroma_doc_stems(collection) -> set[str]:
    metas = collection.get(include=["metadatas"])["metadatas"]
    return {m["doc_id"].lower() for m in metas if m.get("doc_id")}


def gold_source_docs(row: dict) -> list[str]:
    docs = list(row.get("source_docs") or [])
    if row.get("source_doc"):
        docs.append(row["source_doc"])
    return docs


indexed = chroma_doc_stems(col)
all_cids = {m.get("chunk_id") for m in col.get(include=["metadatas"])["metadatas"]}
rows_cov = []
for q in wrapper["queries"]:
    docs = gold_source_docs(q)
    missing = [d for d in docs if Path(d).stem.lower() not in indexed]
    chunk_id = q.get("source_chunk_id") or q.get("legacy_chunk_hint")
    chunk_ok = (chunk_id in all_cids) if chunk_id else None
    rows_cov.append({
        "id": q["id"],
        "eval_mode": q.get("eval_mode"),
        "source_doc_ok": not missing,
        "missing_docs": ", ".join(missing) if missing else "",
        "chunk_id": chunk_id or "",
        "chunk_in_chroma": chunk_ok,
    })

cov_df = pd.DataFrame(rows_cov)
display(cov_df)
n_missing = int((~cov_df["source_doc_ok"]).sum())
print(f"Actas gold ausentes en Chroma: {n_missing}")
if n_missing:
    print("→ Q4/Q13: acta 243591 fuera del índice v2 (eval_mode=skip)")

,id,eval_mode,source_doc_ok,missing_docs,chunk_id,chunk_in_chroma
0,Q1,single_doc,True,,,None
1,Q2,single_doc,True,,,None
2,Q3,single_doc,True,,,None
3,Q4,single_doc,False,243591-DO-AVO-11-V07-251210.docx,,None
4,Q5,single_doc,True,,,None
5,Q6,single_doc,True,,,None
6,Q7,single_doc,True,,,None
7,Q8,single_doc,True,,,None
8,Q9,single_doc,True,,,None
9,Q10,single_doc,True,,,None


Actas gold ausentes en Chroma: 2
→ Q4/Q13: acta 243591 fuera del índice v2 (eval_mode=skip)
